In [0]:
%run ./00_functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from datetime import datetime

In [0]:
test_schema = StructType([
    StructField('transactionID', StringType(), True),
    StructField('dateTime', TimestampType(), True),
    StructField('product', StringType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('unitPrice', DoubleType(), True),
    StructField('totalPrice', DoubleType(), True),
])

test_data = [
    ('T001', datetime(2026, 1, 1, 10, 0), 'Cookie', 2, 3.0, 6.0),
    ('T002', datetime(2026, 1, 1, 11, 0), 'Bread', -1, 4.0, 4.0),
    ('T003', datetime(2026, 1, 1, 12, 0), 'Cake', 2, 5.0, 15.0),
    ('T004', datetime(2026, 1, 1, 13, 0), 'Water123', 4, 5.0, 5.0)
]

test_df = spark.createDataFrame(
    test_data,
    schema=test_schema
)

In [0]:
test_r = validate_transactions(test_df)
valid_df, invalid_df = split_valid_and_rejected(test_r)

r = {
    row['transactionID']: (
        row['dq_status'],
        row['dq_reason']
    )
    for row in test_r.select(
        'transactionID',
        'dq_status',
        'dq_reason'
    ).collect()
}

assert r['T001'] == ('VALID',None)
assert r['T002'] == ('INVALID','Invalid_Quantity')
assert r['T003'] == ('INVALID','Invalid_TotalPrice_sum')
assert r['T004'] == ('INVALID','Invalid_Product')

assert valid_df.count() == 1
assert invalid_df.count() == 3
assert (test_r.count() == valid_df.count() + invalid_df.count())